In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f"Using GPU")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
  print(f"Using MPS")
else:
  device = torch.device("cpu")
  print(f"Using CPU")

Using GPU


In [3]:
train_transforms = transforms.Compose(
  [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.RandomRotation(degrees=15),
  ]
)

test_transforms = transforms.Compose(
  [
    transforms.Resize((244, 244)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
  ]
)

In [12]:
full_dataset_for_train = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=train_transforms
)

full_dataset_for_test = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=test_transforms
)

idxs = np.arange(len(full_dataset_for_train))
labels = full_dataset_for_train.targets

train_idx, test_idx = train_test_split(
  idxs,
  test_size=0.2,
  stratify=labels
)

train_dataset = torch.utils.data.Subset(full_dataset_for_train, train_idx)
test_dataset = torch.utils.data.Subset(full_dataset_for_test, test_idx)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=2, shuffle=False)

In [13]:
np.sum(np.array(labels) == 0), np.sum(np.array(labels) == 1)

(np.int64(93), np.int64(35))

In [14]:
np.sum(np.array(labels) == 0) / len(labels), np.sum(np.array(labels) == 1) / len(labels)

(np.float64(0.7265625), np.float64(0.2734375))

In [6]:
model = nn.Sequential(
  nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, padding="same"),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding="same"),
  nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding="same"),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding="same"),
  nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding="same"),
  nn.MaxPool2d(kernel_size=2),
  nn.AdaptiveAvgPool2d(output_size=1),
  nn.Flatten(),
  nn.Linear(in_features=256, out_features=128), nn.ReLU(),
  nn.Dropout(0.5),
  nn.Linear(in_features=128, out_features=64), nn.ReLU(),
  nn.Dropout(0.5),
  nn.Linear(in_features=64, out_features=1)
).to(device)


In [7]:
m = nn.AdaptiveAvgPool2d(1)
input = torch.tensor([[[[1, 2], [3, 4]], [[5, 6], [7, 8]], [[9, 10], [11, 12]]], ], dtype=torch.float16)
m(input)

tensor([[[[ 2.5000]],

         [[ 6.5000]],

         [[10.5000]]]], dtype=torch.float16)

In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 40

In [9]:
model.train()

for epoch in range(n_epochs):
  running_loss = 0.0

  for i, data in enumerate(train_loader):
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

    optimizer.zero_grad()

    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  
  print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.8415065396065805
Epoch 2, Loss: 0.9585912522732043
Epoch 3, Loss: 0.9225138630352768
Epoch 4, Loss: 0.6204384769879135
Epoch 5, Loss: 0.6998049380148158
Epoch 6, Loss: 0.7727890289297291
Epoch 7, Loss: 0.6861737014031878
Epoch 8, Loss: 0.6815732361054888
Epoch 9, Loss: 0.7121481984561565
Epoch 10, Loss: 0.6979250919585135
Epoch 11, Loss: 0.6034987709101509
Epoch 12, Loss: 0.6537371324557885
Epoch 13, Loss: 0.6871174497931611
Epoch 14, Loss: 0.7515049389764374
Epoch 15, Loss: 0.7165690200293765
Epoch 16, Loss: 0.6515091716074476
Epoch 17, Loss: 0.5776297057084009
Epoch 18, Loss: 0.6954528595886978
Epoch 19, Loss: 0.6272287246058968
Epoch 20, Loss: 3.821227498426571
Epoch 21, Loss: 50.0289084759413
Epoch 22, Loss: 6.398039679905125
Epoch 23, Loss: 3.3972448773828208
Epoch 24, Loss: 0.7950655958231758
Epoch 25, Loss: 1.4241949845762814
Epoch 26, Loss: 0.983310646870557
Epoch 27, Loss: 0.7757120541497773
Epoch 28, Loss: 0.5990847466038722
Epoch 29, Loss: 0.6519423197297489

In [10]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
  for data in test_loader:
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)
    outputs = model(inputs)
    predictions = torch.sigmoid(outputs) > 0.5
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

print(all_predictions)
print(all_labels)
print(np.sum(np.array(all_predictions) == np.array(all_labels).astype(bool)) / len(all_labels))

[array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False]), array([False])]
[array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=floa